In [1]:
%load_ext autoreload
%autoreload 2
%cd /home/abraham/uni/ikt453/project/v1

/home/abraham/uni/ikt453/project/v1


In [2]:
from collections import defaultdict
from copy import deepcopy
from uuid import uuid4

from src.utils import disk
from src.utils import debug

In [3]:
pr = 'data/samples/prospects.json'
dr = 'data/samples_nbaapi/drafts.json'
dh = 'data/samples_nbaapi/drafthistory.json'

pr = disk.read_json(pr)
dr = disk.read_json(dr)
dh = disk.read_json(dh)

In [16]:
from src.processing.ids import id_mapper, IDMapper

In [17]:
def construct_college_team_dimension(pr: dict, id_mapper: IDMapper) -> tuple[list, list, list, dict]:
    # TODO: Ignored fields draft/venue, draft/start_date, draft/end_date, draft/league

    pr = deepcopy(pr)
    
    list_of_prospects: list[dict] = pr['prospects']

    draft_year = pr['draft']['year']

    college_teams = []
    prospects = []

    # --------------------------
    # Helper function to construct entries for conference, division, and team
    # --------------------------
    def to_entry(namespace: str, prefix: str, obj: dict, mappings: str, provider: str, provider_id: str) -> dict:
        entry = {}
        alias = obj.get('alias')
        kwargs = {k: obj.get(k) for k in mappings}

        if alias:
            obj_id = id_mapper(namespace, alias, provider, provider_id, str(uuid4()))
            entry = {
                f'{prefix}_id': obj_id,
                f'{prefix}_alias': alias,
                **{f'{prefix}_{k}': v for k, v in kwargs.items()}
            }

        return entry

    for prospect in list_of_prospects:
        # --------------------------
        # Construct entries for conference, division, and team
        # --------------------------
        conference: dict = prospect.pop('conference', {})
        division: dict = prospect.pop('division', {})
        team: dict = prospect.pop('team', {})

        conference_entry = to_entry(
            namespace='conference',
            prefix='conference',
            obj=conference,
            mappings=('name', ),
            provider='sportsradar',
            provider_id=conference.get('id'),
        )
        
        division_entry = to_entry(
            namespace='division',
            prefix='division',
            obj=division,
            mappings=('name', ),
            provider='sportsradar',
            provider_id=division.get('id'),
        )

        team_entry = to_entry(
            namespace='team',
            prefix='team',
            obj=team,
            mappings=('name', 'market'),
            provider='sportsradar',
            provider_id=team.get('id'),
        )

        college_teams.append({
            **team_entry, 
            **division_entry, 
            **conference_entry
        })

        # --------------------------
        # Construct prospect entry
        # --------------------------
        prospect.pop('source_id', None)

        college_team_entry = college_teams[-1]

        birthplace = prospect.pop('birth_place', '').split(',')
        birthplace = zip(('city', 'state', 'country'), birthplace)
        birthplace = {f'birth_{k}':v.strip() or None for k, v in birthplace}
        
        p_id = id_mapper('prospect', prospect['name'], 'sportsradar', prospect.pop('id'), str(uuid4()))
        
        prospects.append({
            'prospect_id': p_id,
            'draft_year': draft_year,
            **prospect,
            **birthplace,
            'team_id': college_team_entry.get('team_id'),
        })

    return college_teams, prospects



In [18]:
college_teams, prospects = construct_college_team_dimension(pr, id_mapper)

In [19]:
len(college_teams), len(pr['prospects'])

(121, 121)

In [20]:
# debug.prettyprint(prospects, ensure_ascii=False)
debug.prettyprint(id_mapper.ids, ensure_ascii=False)

{
    "conference": {
        "BIG10": {
            "sportsradar": "a30fe8ff-82d2-4521-bc8d-e08e6a9dbb52",
            "custom": "9fee523c-714a-4375-9e36-b64ea2906dad"
        },
        "SEC": {
            "sportsradar": "9ed9d01e-977c-4ba7-ac7d-64035039461c",
            "custom": "ba010e49-0976-4d78-9bc6-e430b367cbc3"
        },
        "BIG12": {
            "sportsradar": "2853cf4d-6d62-4ec6-8e2c-d69f7a01a557",
            "custom": "7aef958f-4425-406b-9bba-9c88cbc77b06"
        },
        "ACC": {
            "sportsradar": "88368ebb-01fb-44d5-a6c6-3e7d46bb3ab7",
            "custom": "4737351c-a153-4c19-90e3-751073dc4a44"
        },
        "MWC": {
            "sportsradar": "93a776e4-d390-48e1-95bb-74945457366a",
            "custom": "9fff4609-b60f-47ce-b881-990e44e8575a"
        },
        "BIGEAST": {
            "sportsradar": "d07bc93e-c84c-44a9-a99d-c213bd0014d6",
            "custom": "d447fcf3-b42b-47f9-a09e-f2d14b6ee05e"
        },
        "IVY": {
            "spor

In [9]:
import pandas as pd
df = pd.DataFrame(college_teams)
df.head()

,team_id,team_alias,team_name,team_market,division_id,division_alias,division_name,conference_id,conference_alias,conference_name
0,ea0268d0-f801-4f83-ba8f-a4eb25898e04,NEB,Cornhuskers,Nebraska,248bff3e-84cf-4317-897d-fe27e608b6c7,D1,NCAA Division I,949bed95-58b2-40ff-ac1c-1c0c859bbb06,BIG10,Big Ten
1,da328854-c2b3-4d17-9c6e-9a2de66df143,OKLA,Sooners,Oklahoma,248bff3e-84cf-4317-897d-fe27e608b6c7,D1,NCAA Division I,00d87edf-9096-4b6e-aaca-8f642801ebc1,SEC,Southeastern
2,0be7fdae-0f70-4188-8499-ef08792308b8,MIZZ,Tigers,Missouri,248bff3e-84cf-4317-897d-fe27e608b6c7,D1,NCAA Division I,00d87edf-9096-4b6e-aaca-8f642801ebc1,SEC,Southeastern
3,f844b44a-2305-49f9-b591-e283ad4dafe0,ALA,Crimson Tide,Alabama,248bff3e-84cf-4317-897d-fe27e608b6c7,D1,NCAA Division I,00d87edf-9096-4b6e-aaca-8f642801ebc1,SEC,Southeastern
4,2c7d0198-54d7-475f-9224-6960e8e7aba7,UK,Wildcats,Kentucky,248bff3e-84cf-4317-897d-fe27e608b6c7,D1,NCAA Division I,00d87edf-9096-4b6e-aaca-8f642801ebc1,SEC,Southeastern


In [10]:
df.conference_name.value_counts()

conference_name
Southeastern         34
Big Ten              18
Atlantic Coast       15
Big 12               12
Big East             10
West Coast            3
American Athletic     3
Mountain West         2
Atlantic 10           2
Summit League         1
Ivy                   1
Sun Belt              1
Conference USA        1
Name: count, dtype: int64

In [13]:
# debug.prettyprint(dh, ensure_ascii=False)
list(dh)

dh_df = pd.DataFrame(dh['resultSets'][0]['rowSet'], columns=dh['resultSets'][0]['headers'])
dh_df.SEASON = dh_df.SEASON.astype(int)
subset = dh_df[dh_df.SEASON >= 1000]

In [14]:
team_columns = ['TEAM_ID', 'TEAM_CITY', 'TEAM_NAME', 'TEAM_ABBREVIATION']
organization_columns = ['ORGANIZATION', 'ORGANIZATION_TYPE']
draft_columns = [
    'PERSON_ID', 'PLAYER_NAME', 'SEASON', 'ROUND_NUMBER', 'ROUND_PICK', 'OVERALL_PICK', 
    *team_columns, *organization_columns
]
draft_subset = subset[draft_columns]
draft_subset

,PERSON_ID,PLAYER_NAME,SEASON,ROUND_NUMBER,ROUND_PICK,OVERALL_PICK,TEAM_ID,TEAM_CITY,TEAM_NAME,TEAM_ABBREVIATION,ORGANIZATION,ORGANIZATION_TYPE
0,1642843,Cooper Flagg,2025,1,1,1,1610612742,Dallas,Mavericks,DAL,Duke,College/University
1,1642844,Dylan Harper,2025,1,2,2,1610612759,San Antonio,Spurs,SAS,Rutgers,College/University
2,1642845,VJ Edgecombe,2025,1,3,3,1610612755,Philadelphia,76ers,PHI,Baylor,College/University
3,1642851,Kon Knueppel,2025,1,4,4,1610612766,Charlotte,Hornets,CHA,Duke,College/University
4,1642846,Ace Bailey,2025,1,5,5,1610612762,Utah,Jazz,UTA,Rutgers,College/University
...,...,...,...,...,...,...,...,...,...,...,...,...
8369,79320,Bob Jake,1947,0,0,0,1610610024,Baltimore,Bullets,BAL,Vermont,College/University
8370,79322,Charles Raynor,1947,0,0,0,1610610024,Baltimore,Bullets,BAL,Houston,College/University
8371,79323,John Rusinko,1947,0,0,0,1610610024,Baltimore,Bullets,BAL,Penn State,College/University
8372,76773,Harry Gallatin,1947,0,0,0,1610610024,Baltimore,Bullets,BAL,Truman State,College/University


In [ ]:
draft_subset.ORGANIZATION.value_counts()

In [ ]:
debug.prettyprint(id_mapper.ids, ensure_ascii=False)

In [ ]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()
bearer = os.environ['CFBD_API_KEY']
header = {'Authorization': f'Bearer {bearer}'}
url = 'https://api.collegebasketballdata.com/teams'
rsp = requests.get(url, headers=header)
rsp.status_code

In [ ]:
y = rsp.json()

In [ ]:
y = pd.DataFrame(y)

In [ ]:
xs = set(x.shortName.str.lower().values)
ds = set(df.conference_name.str.lower().values)
len(xs), len(ds), len(xs.intersection(ds))

In [ ]:
ds.difference(xs)

In [ ]:
c = disk.read_json('data/college/college2.json')

In [ ]:
ct = []
for division in c['divisions']:
    for conference in division['conferences']:
        for team in conference['teams']:
            ct.append({
                'team_name': team['name'],
                'team_alias': team['alias'],
                'team_market': team['market'],
                'division_name': division['name'],
                'division_alias': division['alias'],
                'conference_name': conference['name'],
                'conference_alias': conference['alias'],
            })

In [ ]:
ct = pd.DataFrame(ct)

In [ ]:
xss = set(draft_subset.ORGANIZATION.str.lower())
yss = set(ct.team_market.str.lower())
len(xss), len(yss), len(xss.intersection(yss))

In [ ]:
draft_subset

In [ ]:
xss.difference(yss)

In [ ]:
gids = disk.read_json('data/unique_game_ids.json')
gids = gids[-1]['data']
gids = pd.DataFrame(gids)

In [ ]:
gids

In [ ]:
url = 'https://cdn.nba.com/static/json/liveData/boxscore/boxscore_0022000181.json'
rsp = requests.get(url)
rsp.status_code

In [ ]:
dict(rsp.headers)

In [ ]:
tmp = disk.read_json('data/nba/tmp.json')

In [ ]:
disk.write_json('data/nba/tmp.json', tmp)

In [23]:
header_contents = """
accept
*/*
accept-encoding
gzip, deflate, br, zstd
accept-language
nb-NO,nb;q=0.9,no;q=0.8,nn;q=0.7,en-US;q=0.6,en;q=0.5
cache-control
no-cache
cookie
mediakindauth2token=AuthToken1fZDLbtswFES_xtoUFsSnyIUWQuJHUrhPF0G7KWjy2iYskSpJxXa_vpITt0iRFiAuiZkzl8DUvbHgNFT7lLo4IfUEz4cToDlbt-tUSOe8C96o3G1Unh6Jbnxvcu3bgcruYuwh_JWNKf4nMrJ_UL2HVsX81DbRqy73YTeoxzgMXBRsuC7DGnDJpvPw1I2y7eg71UK16yGm7wa2qm9StvYHcOtzB9Vi1LM1OOXSnamuwC08Wg0X4gE2Wd00_gjmisXf3JcIYYi93F5r7fvLupf67dP9auYTqKatduAgDB870w9QfL2C1urgo9-m56KuJYixBP6PEoJvnkt4M67O6j7tfbA_VbLerVQ8VIQIVEpRFCUTnFGJOOeMMymxKIkUo4swFwQVSGBGMZKc0lIwyRBjBR0pKigrJRcMYy4Z4XjwC5HNTp0NEN-7CpWloMWgF9lNAJXAPImsxKORvYXzorem0gYVDEkz5eWWTalkaiqoVFNEStC65EC0ypar-ubzssaMV2zJrTuR7sPq3hv8rjt8my24P-wf9PzrVn28p4pO8OYHzGdHQifE_AI; nbatag_main_v_id=019d71a08e8c0021001940bbf3e60506f0019067019c0; OptanonAlertBoxClosed=2026-04-09T11:13:49.151Z; eupubconsent-v2=CQiZB7AQiZB7AAcABBENCZFsAP_gAAAAACiQMSgB4CIEQSFBACJwAIoAAAAEQAAAAEAAAAABAAAAAAAABAQAECAAAACAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAIAAAAAAAAAAAAAAIAAAAAAAAAAAAAAAEgAAAABIAAAAAAAAAAAARKAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAATyff7Pn__rl_e7X_ve_n3zv8oXH77r____f_-7___2b_-___b-__7JoAAACQkAYACoAIIAZABoAEwAQgC8wgAIBR4DFh0AcABYAFQAQQAyADQAJgBFgF5jgAQBCAGLEIAIACyUAMABYATAF5kgAIDFikAcABYAFQAQQAyADQAJgBFgF5lAAIDFgAAA.f_wAAAAAAAAA.IMSwRYAFAAaABUADIAIAASAAqABaADIAGgAOgAigBJgCYAJwAWwAvgBhAD8AIAAQgApABlAEAAIQARYAjoBOwEagKPAXgAvMBiwDGQGfANmAbUA20Bt4DcwJwQTjBOYCdME7ATwgnkCfME-wUXgoyCjkFHgUmgpQClcFLQUvgpiCmQFNIKbAqJBUYFSYKlgqxBVoFXoKwAraBW8CuIFcwK7QV4BXoCvkFfgWJgsWCx8FkQXagu4DDMGGwYcgw8DEEGIgYjAxKAAA; at_check=true; s_ecid=MCMID%7C81492290038141589478614812525550188763; s_cc=true; AMCVS_248F210755B762187F000101%40AdobeOrg=1; __tld=nba.com; canPersist=true; AMCV_248F210755B762187F000101%40AdobeOrg=179643557%7CMCMID%7C81492290038141589478614812525550188763%7CMCAID%7CNONE%7CMCOPTOUT-1775762150s%7CNONE%7CvVersion%7C5.5.0%7CMCIDTS%7C20553%7CMCAAMLH-1776359750%7C6%7CMCAAMB-1776359750%7Cj8Odv6LonN4r3an7LhD3WZrU1bUpAkFkkiY1ncBR96t2PTI; _gid=GA1.2.470863044.1775754997; _ga_XQNG16PXVR=GS2.1.s1775754943$o1$g1$t1775755005$j60$l0$h0; _ga=GA1.2.2059718217.1775754944; _hjSessionUser_2837354=eyJpZCI6ImI1Yzg4MmFhLTc2OGYtNWQwZC04ZGRmLTM0ZTBjMjJiZTRkOSIsImNyZWF0ZWQiOjE3NzU3NTQ5NDg2ODIsImV4aXN0aW5nIjp0cnVlfQ==; bm_mi=CF07E3618B6A993993AB2F8A7CC2019F~YAAQXnchF/O4k2KdAQAABI5ldR/drkmM/Rjj/2IcNYZbVoU7bc9WSKReANtdK32kCaZzPaTR7s4xEpn0yx4icbh/+tUw30vcbkgw1Bd+a5ITz8tBmNfAOU28DAoWF68U9dcTLPhi4LSD17BAOYCqqmBL15pw9zw/OyWYCFMtAqwXypN/FGOMbvxbHPN5ZuWbsdkx4mYHXfXVt6yrWBJGoIEAlWMbVCyESS+FZjHJ0CDxKpBd8FQGDvZIePNP7afLOh0az6/wX36hGLNcb/WORvCvR3JRJu85ytOWBGshzgG+IKp5fqHM8JndHPPplR4=~1; nbatag_main__sn=7; nbatag_main_ses_id=1775791084179%3Bexp-session; nbatag_main_dc_visit=6; nbatag_main__ss=0%3Bexp-session; nbatag_main_dc_region=eu-central-1%3Bexp-session; ab.storage.deviceId.cf150dab-3153-49b0-b48c-66a7c18688ea=%7B%22g%22%3A%22a14abc75-c221-e8a4-aff0-92da6118005a%22%2C%22c%22%3A1775733229794%2C%22l%22%3A1775791084766%7D; AMP_MKTG_2442d50754=JTdCJTIycmVmZXJyZXIlMjIlM0ElMjJodHRwcyUzQSUyRiUyRnd3dy5nb29nbGUuY29tJTJGJTIyJTJDJTIycmVmZXJyaW5nX2RvbWFpbiUyMiUzQSUyMnd3dy5nb29nbGUuY29tJTIyJTdE; _abck=F7C1C43FA6297958E1CF4325D30CA083~0~YAAQXnchF/LZk2KdAQAAMdh9dQ9cgRRby2YJh3OZJS3u5SWWGcqlfchbwazOPT5wG+bdKg1D/WWvx/8uEVv/lLQO2ZVJ+haMzyf8ndo1hG34G+zyk7qhLBtfXfoRTZfFVPQicMX/buRtZFyiEKuNDMKNw4QMcsvEp5hUPm8Ca5EG3WFZmzXZt5U9WWuP0ohytf27wxYQGyYJrr4I0Xftj+P4wewIfxkzQlrhks3wmpQChVdAUE+hx1JOhJM/jW10KSnPqyzrXcyQmwAQrX2gSA7N324vZraLlg71r2Nt0le5soR3fJXOj+9qJ7riPCz2D5kyith1PRphVQ+o2INxllwX0X+3qkGuSmo+NPUxAAYBnT0sfGlYHgX6CNw4n1aVWCljqeR38ZtF4D0LM6KEit9MpCjB7aq7WYGiafm79dFPOAzKZ52iVhKop24iq36T7F/Wni5eBorYkD2h2z651RQEPoQQ5YSkeayZzOBHBSpkTRIecF+S42IvVtHMTZKdtBnyFVbG4ggSW4ySncYH5buXCO39LK6H6dDVOU4l5XtJvad4rdNzCSvSSh2XKJL4GGtuVYeDAm4kTbJS2O1XR18fDHUiG5gB8fkB2UNcNHQk/Kw/gJ6ivIM0HeLY1Th/uKhariIDddzu0UYtW1ZoxSqL7g5L1ZfS6eoLV35I92Tq7a1XlDpFWE0yM1aRUt9Fb1YDp9JuxxrFg2/cGJeZzhdmrmXLRWkstiHNVIJhDIkYiYe5q+GgycTEJGgyDW0FC3wSoLwoOks=~-1~-1~1775794683~AAQAAAAF%2f%2f%2f%2f%2fzi6Oro65XKwSRk1AVM6LDhBxMVqKRXr7T3LIAIA1tJQTnqI4TQfVhFSaXkqCMHZyqpGBG+n6INpDVa41WEJo0vWNUfByE3K8cOh5FXLcvu35DH+a8R9O%2fJVjouMl6zvTiOjHX4FtwLDN%2fMhmB2C9fZzJO5Kfxm7WR9ExPLORWiCbPe9detukfiEX2OT1RwtDG3DXvUtDPZooSjRTz6D30tEDNdX1BLQH74eGKz5XYkmIUM%3d~-1; iframeRef=www.nba.com/game/mem-vs-bkn-0021300908/game-charts; s_gpv_pageModal=nba%3Agames%3Agame-details%3Agame-charts; aaCustPrevPage=nba:games:game-details:game-charts; nbaOrigin=Page%20View%3A%20Game%20Details%20Box%20Score%7C%7CGames; _cs_mk_aa=0.549901771393414_1775792883513; bm_ss=ab8e18ef4e; bm_so=ED14246E7464DAC1E71C968467B9945A754F4570EF4D5F3AD62588A2E0C69063~YAAQXnchF9Dmk2KdAQAAtQ6FdQcKDcjEUzVBqYlksJXxJ/StBr1mU4ji/a0y27KW6USRYbrDVggaZB+2k0K7TgycqiKAIl7KkZi8Ghknm4FHdvhgnc4yim+hfyZcQZR6UlUA0oE6tkytVz+K4FsqtMqEBqQMy3SOo6lKwb8i7kSqerstuKTj3wB5gz+YEOl3wYNIl+7SWIyAtwGXMkRANXF18C121isOsEkFpK2NyEJj0APasiIJOeirLRqTinNStXM/HYjjvCJU5FmYrto0iLpjzK3eoJ69bD77az8NhTnT3wB+f/OQVfXp17Jyp/vgLN+Yd6EJdL+HY6qnRHXX5oQs7WaKIGgdCk+02lfj9VUxcouOSmSY5ulBViMjmUIeHgAZHYhlchYOw/t9EXYsE+6e4RO+WbXSCPib01P+MS7CVJ7I6Z3RqjWkuJe8yo7wRCK4BIrP1p5Ur/llTDiclIcojUIOewodYb1pmqiajSMIHeB1Pqzb; OptanonConsent=isGpcEnabled=0&datestamp=Fri+Apr+10+2026+05%3A52%3A28+GMT%2B0200+(Central+European+Summer+Time)&version=202511.1.0&browserGpcFlag=0&isIABGlobal=false&hosts=&consentId=f5564860-72ae-4a15-8ab6-602e8a51a37d&interactionCount=2&isAnonUser=1&landingPath=NotLandingPage&groups=dsa%3A1%2Ccad%3A1%2CNBAad%3A1%2Cmcp%3A1%2CNBAmt%3A1%2Cpad%3A1%2Cpap%3A1%2Cgld%3A1%2Cpcd%3A1%2Cpcp%3A1%2Cmap%3A1%2Cmra%3A1%2Cpdd%3A1%2Csid%3A1%2Csec%3A1%2Ctdc%3A1%2Ccos%3A1%2Cdlk%3A1%2Cdid%3A1%2Cdsh%3A1%2Cdsl%3A1%2Cven%3A1%2Creq%3A1&AwaitingReconsent=false&intType=1&geolocation=NO%3B03; nbatag_main__pn=9%3Bexp-session; nbatag_main__se=22%3Bexp-session; nbatag_main__st=1775794948815%3Bexp-session; s_ips=1304; s_ppv=nba%253Agames%253Agame-details%253Agame-charts%2C29%2C29%2C1304%2C1%2C3; __gads=ID=9d31533db7910237:T=1775733229:RT=1775793148:S=ALNI_MbqHDWjDYMzIk1F-NyEdcVAxuMRkA; __gpi=UID=0000139a73d9ab7f:T=1775733229:RT=1775793148:S=ALNI_MbUBVKh9VfIsZqhwnOtLD4sZqN7xQ; __eoi=ID=1c65af6dd079bd46:T=1775733229:RT=1775793148:S=AA-AfjYaQbbstDhV2gY1e295P99o; nbatag_main_dc_event=22%3Bexp-session; ab.storage.sessionId.cf150dab-3153-49b0-b48c-66a7c18688ea=%7B%22g%22%3A%22a038162f-24a6-e827-720e-1163acb0f539%22%2C%22e%22%3A1775794949061%2C%22c%22%3A1775791084763%2C%22l%22%3A1775793149061%7D; AMP_2442d50754=JTdCJTIyZGV2aWNlSWQlMjIlM0ElMjIwMTlkNzFhMDhlOGMwMDIxMDAxOTQwYmJmM2U2MDUwNmYwMDE5MDY3MDE5YzAlMjIlMkMlMjJ1c2VySWQlMjIlM0ElMjIlMjIlMkMlMjJzZXNzaW9uSWQlMjIlM0ExNzc1NzkxMDg1OTYwJTJDJTIyb3B0T3V0JTIyJTNBZmFsc2UlMkMlMjJsYXN0RXZlbnRUaW1lJTIyJTNBMTc3NTc5MzE0OTk0MyUyQyUyMmxhc3RFdmVudElkJTIyJTNBMTQ1JTJDJTIycGFnZUNvdW50ZXIlMjIlM0EwJTdE; s_tp=4442; bm_lso=ED14246E7464DAC1E71C968467B9945A754F4570EF4D5F3AD62588A2E0C69063~YAAQXnchF9Dmk2KdAQAAtQ6FdQcKDcjEUzVBqYlksJXxJ/StBr1mU4ji/a0y27KW6USRYbrDVggaZB+2k0K7TgycqiKAIl7KkZi8Ghknm4FHdvhgnc4yim+hfyZcQZR6UlUA0oE6tkytVz+K4FsqtMqEBqQMy3SOo6lKwb8i7kSqerstuKTj3wB5gz+YEOl3wYNIl+7SWIyAtwGXMkRANXF18C121isOsEkFpK2NyEJj0APasiIJOeirLRqTinNStXM/HYjjvCJU5FmYrto0iLpjzK3eoJ69bD77az8NhTnT3wB+f/OQVfXp17Jyp/vgLN+Yd6EJdL+HY6qnRHXX5oQs7WaKIGgdCk+02lfj9VUxcouOSmSY5ulBViMjmUIeHgAZHYhlchYOw/t9EXYsE+6e4RO+WbXSCPib01P+MS7CVJ7I6Z3RqjWkuJe8yo7wRCK4BIrP1p5Ur/llTDiclIcojUIOewodYb1pmqiajSMIHeB1Pqzb~1775793151063; ak_bmsc=CAD8DB6E9BD11AAABEED71609463010A~000000000000000000000000000000~YAAQXnchF3vnk2KdAQAA2k+FdR+C3TPvBHlHSsYmaieymDLwVTlAFdiiTiwLfyIG+s9AGt4UElmtNAO39XK1837v5oQr2XKNcmG6Sok99I0xeN1zqM41SCA1LrdEoQxx6DTBu0nhSNL9nXg3VgkRcW1zzFgjQPdux+7hWsd6C2zX1Zuq2F1oM+H4duG+AwJV7xVVI55PW2W/8PyKkgdK0yOPtpqT0xH0Gb4BA0ikT3VeGVzVBbcToEjRTCEWqMqtek70LykB6tslsQyrjP3ovpGg45c8/cfEn96qoOzFybskLcONxtwR1ZvGJc8OgT4GVmXSn9VMSO8fLq8HaNnaJK2QB43/iUYQhD+U2B8LOqtOmJB1khQiVLZAckBzFK9yOd5JyrlyXGgNcs8RXRHyza1IrbLSa/hev+9Q+W0Yww2YF4tRcwqM9EySpmjm8epaPzp5U2ZQPTI6tEteEH9QBpWzO25kaoiSrUw3fkZUzxTRPNTgTvHiiDEDfGQ5I1Au+wwwFAXXV20cXw==; bm_s=YAAQXnchF3znk2KdAQAA20+FdQVbIm53nio3fO/YvHyErlX7FaCO3Sw26uD2uR9x6SuUHkzMDzRLv/zwmC83wsvjD2FTdLVlXU8sS2mpeimnjAxmUpKqOC+iV3iXj4HNtTwu8cw8vvTUr0UoTQZS5cljXhPh5Kerbd3xRXrWYXdwjMlX7x955jbLaSWql2Eb6v2hn64vbWqcCEdG7jL915fhB4FKk70fEJZSz0dvrUJn9kwQDenrV6C4jCeAEkYWYz5CTDxmDjP33WOkKb7ClaNNSMP7+O07WnPQfTHw+VcJto5VpQvJE6fMghejmTZhLPxWahV2qmtbDNdqczDD+9xvwl1/mq23nVb/SKkCVmWsAdKWywLej7Yyt5Efmpc8JhHx/Maq8yf8peefwdS8YSU/m72dz6DUf7scQjQrP9UQpNNybwzQwrhQ9IDVB1vnYvduMGMFAewjSuzhiZYH8Mhzqa+lNFNp1ciwnqEbW2zb8QQcu2XxxcKoe/NF5JU+f0LYacJoaSrdUQ1c7q9S5ZaRi42JDzJxnP/j8BOnIg+6WOsvdFSC0uXvXP48k3BfrM9Dm0zP++BidDQfqSXRuufQ03C/1g308BIeui7Xc42lstQvlyfCzLeSrSlYEoQohQu6YQhPMlhNFe7tX1n0gmFyXittwI1k6OznwCKyxOcaGRZ933gIS76DH4QpleT+YW1NJT59Tgnv7Ar8xO4kQ0deMUXLMQmv1zppi38qOMOKxC+XSRXGllPoJcRTj4jfuMK0EWPgXgW2lxsj9Aoq5fxrOzqEu4FT+lcWi2G6D0awd+5ykCUgLDiRk5UyPux1ROQisI6KWOA7AOdswJv0PwpikGu9rKpj4ieOJ2J7ZORVHiJw868dHZERI1ESgn4UlPmg6Xd5mtt0NVkE20YK71xqYB3H1RJxycax6YskFEXcywWWQWSioHrPiQcNQdLuPX3wDs+9VtXrAAX/HGW4PFvh96UXx6fvOenNVCDkNO1bjc5l7HvNkoL3y1s=; bm_sv=A234BD4DCCCBF7F480613BC4F4B4D628~YAAQXnchF33nk2KdAQAA20+FdR+jhyk3nHtiNFDobns89KfC/LOeLq4gFy9zwTDBctdYPM885c8AFdEvFKDPPeKhe7T96SR3FWOOF9PaYT4eNMoMMk15oSdxnEeVdfMYWW5CRTTAHkkdQ8c4vmrZvXDpy6X8GuIhsWTr2Wau8216mrN3PMJXvf4I3gGe8YhU9iApeJcRZBwvbEpMD5NCXK4IP38vcAOHdW19N2Df3YIojoMbdbVsqKOVSDc8uQ==~1; bm_sz=B50DBEF5E27D8777BCA40535DD907D51~YAAQXnchF37nk2KdAQAA20+FdR+6KWK8iG0rVZCjaa54/6VFEGXEmfMby28hIW97UFZL9JxmaYr4AKsHkTP3aPYo+WUagMWIeBBnqukxv8pTtUu4bw5lEbP0q9p2rO/tRjZ2BLZoEe35X/5bKC4Rcfqe2HekR7VPkyEtSn/RQt89KunBGtnECdsBjNNEAssGJqcqr7CjVzZ20xUuQognexthovA+LujBoX0o38L07Qs4lfueEhBoVbuAns5hrhDWRDy8GuyRClNGPSJI18oU289bNCmA3igddeOQbX0GnDisWRJcLaNvBWE8hcMu2j0p3Tsq/UVg2oUy9r60oo/C8CFPmCOzi8QY6RUJsI3tIHOem9A4s172Ht43DFenDKPpU5brE/pNcWi3MqiDwxgP/VZaZtispt7+QRQiDyIThiwo26bPfCj9NhD7wobcPI1qSbFpdZZRFMnilviIybmnUoYHxk2WFirTRBju9qHDr7AZ+Q/rF42ippbFEjRhDRW1aQ==~4277301~3552825
pragma
no-cache
referer
https://www.nba.com/game/mem-vs-bkn-0021300908/game-charts
sec-ch-ua
"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"
sec-ch-ua-mobile
?0
sec-ch-ua-platform
"Windows"
sec-fetch-dest
script
sec-fetch-mode
no-cors
sec-fetch-site
same-origin
user-agent
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36
""".strip()

header_contents = """
accept
*/*
accept-encoding
gzip, deflate, br, zstd
accept-language
nb-NO,nb;q=0.9,no;q=0.8,nn;q=0.7,en-US;q=0.6,en;q=0.5
cache-control
no-cache
origin
https://www.nba.com
pragma
no-cache
priority
u=1, i
referer
https://www.nba.com/
sec-ch-ua
"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"
sec-ch-ua-mobile
?0
sec-ch-ua-platform
"Windows"
sec-fetch-dest
empty
sec-fetch-mode
cors
sec-fetch-site
same-site
user-agent
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36
""".strip()

headers = {}
for idx, line in enumerate(header_contents.splitlines()):
    if idx % 2 == 0:
        key = line.strip()
    else:
        value = line.strip()
        headers[key] = value

In [ ]:
url = 'https://www.nba.com/game/mem-vs-bkn-0021300908/game-charts'
rsp = requests.get(url, headers=headers)
rsp.status_code

In [53]:
import requests
url = 'https://stats.nba.com/stats/playerindex?College=&Country=&DraftPick=&DraftRound=&DraftYear=&Height=&Historical=1&LeagueID=00&Season=2025-26&SeasonType=Regular%20Season&TeamID=0&Weight='
url = 'https://stats.nba.com/js/data/playermovement/NBA_Player_Movement.json'
url = 'https://stats.nba.com/stats/leaguegamelog?Counter=1000&DateFrom=&DateTo=&Direction=DESC&ISTRound=&LeagueID=00&PlayerOrTeam=P&Season=2024-25&SeasonType=Regular%20Season&Sorter=DATE'
rsp = requests.get(url, headers=headers)
rsp.status_code

200

In [44]:
x = {
    "name": "otFlat",
    "html": "PGRpdiBpZD0ib25ldHJ1c3QtYmFubmVyLXNkayIgY2xhc3M9Im90RmxhdCI+PGRpdiByb2xlPSJkaWFsb2ciPjxkaXYgY2xhc3M9Im90LXNkay1jb250YWluZXIiPjxkaXYgY2xhc3M9Im90LXNkay1yb3ciPjxkaXYgaWQ9Im9uZXRydXN0LWdyb3VwLWNvbnRhaW5lciIgY2xhc3M9Im90LXNkay1laWdodCBvdC1zZGstY29sdW1ucyI+PGRpdiBjbGFzcz0iYmFubmVyX2xvZ28iPjwvZGl2PjxkaXYgaWQ9Im9uZXRydXN0LXBvbGljeSI+PGgyIGlkPSJvbmV0cnVzdC1wb2xpY3ktdGl0bGUiPlRpdGxlPC9oMj48ZGl2IGlkPSJvbmV0cnVzdC1wb2xpY3ktdGV4dCI+dGl0bGU8YSBocmVmPSIjIj5wb2xpY3k8L2E+PC9kaXY+PGRpdiBjbGFzcz0ib3QtZHBkLWNvbnRhaW5lciI+PGgzIGNsYXNzPSJvdC1kcGQtdGl0bGUiPldlIGNvbGxlY3QgZGF0YSBpbiBvcmRlciB0byBwcm92aWRlOjwvaDM+PGRpdiBjbGFzcz0ib3QtZHBkLWNvbnRlbnQiPjxwIGNsYXNzPSJvdC1kcGQtZGVzYyI+ZGVzY3JpcHRpb248L3A+PC9kaXY+PC9kaXY+PC9kaXY+PC9kaXY+PGRpdiBpZD0ib25ldHJ1c3QtYnV0dG9uLWdyb3VwLXBhcmVudCIgY2xhc3M9Im90LXNkay10aHJlZSBvdC1zZGstY29sdW1ucyI+PGRpdiBpZD0ib25ldHJ1c3QtYnV0dG9uLWdyb3VwIj48YnV0dG9uIGlkPSJvbmV0cnVzdC1wYy1idG4taGFuZGxlciI+Y2hvaWNlPC9idXR0b24+IDxidXR0b24gaWQ9Im9uZXRydXN0LXJlamVjdC1hbGwtaGFuZGxlciI+UmVqZWN0PC9idXR0b24+IDxidXR0b24gaWQ9Im9uZXRydXN0LWFjY2VwdC1idG4taGFuZGxlciI+QWNjZXB0PC9idXR0b24+PC9kaXY+PC9kaXY+PC9kaXY+PC9kaXY+PGRpdiBjbGFzcz0ib3Qtc2RrLWNvbnRhaW5lciIgaWQ9ImJhbm5lci1vcHRpb25zIj48ZGl2IGNsYXNzPSJvdC1zZGstZm91ciBvdC1zZGstY29sdW1ucyBiYW5uZXItb3B0aW9uIj48YnV0dG9uIGFyaWEtZXhwYW5kZWQ9ImZhbHNlIiBjbGFzcz0iYmFubmVyLW9wdGlvbi1pbnB1dCI+PHNwYW4gY2xhc3M9ImJhbm5lci1vcHRpb24taGVhZGVyIj48aDQ+UHVycG9zZSB3ZSB1c2U8L2g0PjxzcGFuIGNsYXNzPSJvdC1hcnJvdy1jb250YWluZXIiPjwvc3Bhbj48L3NwYW4+PC9idXR0b24+PGRpdiBjbGFzcz0iYmFubmVyLW9wdGlvbi1kZXRhaWxzIj5wdXJwb3NlIGRlc2NyaXB0aW9uPC9kaXY+PC9kaXY+PC9kaXY+PCEtLSBDbG9zZSBCdXR0b24gLS0+PGRpdiBpZD0ib25ldHJ1c3QtY2xvc2UtYnRuLWNvbnRhaW5lciI+PGJ1dHRvbiBjbGFzcz0ib25ldHJ1c3QtY2xvc2UtYnRuLWhhbmRsZXIgb25ldHJ1c3QtY2xvc2UtYnRuLXVpIGJhbm5lci1jbG9zZS1idXR0b24gb3QtY2xvc2UtaWNvbiI+PC9idXR0b24+PC9kaXY+PCEtLSBDbG9zZSBCdXR0b24gRU5ELS0+PC9kaXY+PC9kaXY+",
    "css": "#onetrust-banner-sdk{box-shadow:0 0 18px rgba(0,0,0,.2)}#onetrust-banner-sdk.otFlat{position:fixed;z-index:2147483645;bottom:0;right:0;left:0;background-color:#fff;max-height:90%;overflow-x:hidden;overflow-y:auto}#onetrust-banner-sdk.otFlat.top{top:0px;bottom:auto}#onetrust-banner-sdk.otRelFont{font-size:1rem}#onetrust-banner-sdk>.ot-sdk-container{overflow:hidden}#onetrust-banner-sdk::-webkit-scrollbar{width:11px}#onetrust-banner-sdk::-webkit-scrollbar-thumb{border-radius:10px;background:#c1c1c1}#onetrust-banner-sdk{scrollbar-arrow-color:#c1c1c1;scrollbar-darkshadow-color:#c1c1c1;scrollbar-face-color:#c1c1c1;scrollbar-shadow-color:#c1c1c1}#onetrust-banner-sdk #onetrust-policy{margin:1.25em 0 .625em 2em;overflow:hidden}#onetrust-banner-sdk #onetrust-policy .ot-gv-list-handler{float:left;font-size:.82em;padding:0;margin-bottom:0;border:0;line-height:normal;height:auto;width:auto}#onetrust-banner-sdk #onetrust-policy-title{font-size:1.2em;line-height:1.3;margin-bottom:10px}#onetrust-banner-sdk #onetrust-group-container{position:relative}#onetrust-banner-sdk #onetrust-policy-text{clear:both;text-align:left;font-size:.88em;line-height:1.4}#onetrust-banner-sdk #onetrust-policy-text *{font-size:inherit;line-height:inherit}#onetrust-banner-sdk #onetrust-policy-text a{font-weight:bold}#onetrust-banner-sdk #onetrust-policy-title,#onetrust-banner-sdk #onetrust-policy-text{color:dimgray;float:left}#onetrust-banner-sdk #onetrust-button-group-parent{min-height:1px;text-align:center}#onetrust-banner-sdk #onetrust-button-group{display:inline-block}#onetrust-banner-sdk #onetrust-accept-btn-handler,#onetrust-banner-sdk #onetrust-reject-all-handler,#onetrust-banner-sdk #onetrust-pc-btn-handler{background-color:#68b631;color:#fff;border-color:#68b631;margin-right:1em;min-width:125px;height:auto;white-space:normal;word-break:break-word;word-wrap:break-word;padding:12px 10px;line-height:1.2;font-size:.813em;font-weight:600}#onetrust-banner-sdk #onetrust-pc-btn-handler.cookie-setting-link{background-color:#fff;border:none;color:#68b631;text-decoration:underline;padding-left:0;padding-right:0}#onetrust-banner-sdk .onetrust-close-btn-ui{width:44px;height:44px;background-size:12px;border:none;position:relative;margin:auto;padding:0}#onetrust-banner-sdk .banner_logo{display:none}#onetrust-banner-sdk.ot-bnr-w-logo .ot-bnr-logo{position:absolute;top:50%;transform:translateY(-50%);left:0px;margin-right:5px}#onetrust-banner-sdk.ot-bnr-w-logo #onetrust-policy{margin-left:65px}#onetrust-banner-sdk .ot-b-addl-desc{clear:both;float:left;display:block}#onetrust-banner-sdk #banner-options{float:left;display:table;margin-right:0;margin-left:1em;width:calc(100% - 1em)}#onetrust-banner-sdk .banner-option-input{cursor:pointer;width:auto;height:auto;border:none;padding:0;padding-right:3px;margin:0 0 10px;font-size:.82em;line-height:1.4}#onetrust-banner-sdk .banner-option-input *{pointer-events:none;font-size:inherit;line-height:inherit}#onetrust-banner-sdk .banner-option-input[aria-expanded=true]~.banner-option-details{display:block;height:auto}#onetrust-banner-sdk .banner-option-input[aria-expanded=true] .ot-arrow-container{transform:rotate(90deg)}#onetrust-banner-sdk .banner-option{margin-bottom:12px;margin-left:0;border:none;float:left;padding:0}#onetrust-banner-sdk .banner-option:first-child{padding-left:2px}#onetrust-banner-sdk .banner-option:not(:first-child){padding:0;border:none}#onetrust-banner-sdk .banner-option-header{cursor:pointer;display:inline-block}#onetrust-banner-sdk .banner-option-header :first-child{color:dimgray;font-weight:bold;float:left}#onetrust-banner-sdk .banner-option-header .ot-arrow-container{display:inline-block;border-top:6px solid rgba(0,0,0,0);border-bottom:6px solid rgba(0,0,0,0);border-left:6px solid dimgray;margin-left:10px;vertical-align:middle}#onetrust-banner-sdk .banner-option-details{display:none;font-size:.83em;line-height:1.5;padding:10px 0px 5px 10px;margin-right:10px;height:0px}#onetrust-banner-sdk .banner-option-details *{font-size:inherit;line-height:inherit;color:dimgray}#onetrust-banner-sdk .ot-arrow-container,#onetrust-banner-sdk .banner-option-details{transition:all 300ms ease-in 0s;-webkit-transition:all 300ms ease-in 0s;-moz-transition:all 300ms ease-in 0s;-o-transition:all 300ms ease-in 0s}#onetrust-banner-sdk .ot-dpd-container{float:left}#onetrust-banner-sdk .ot-dpd-title{margin-bottom:10px}#onetrust-banner-sdk .ot-dpd-title,#onetrust-banner-sdk .ot-dpd-desc{font-size:.88em;line-height:1.4;color:dimgray}#onetrust-banner-sdk .ot-dpd-title *,#onetrust-banner-sdk .ot-dpd-desc *{font-size:inherit;line-height:inherit}#onetrust-banner-sdk.ot-iab-2 #onetrust-policy-text *{margin-bottom:0}#onetrust-banner-sdk.ot-iab-2 .onetrust-vendors-list-handler{display:block;margin-left:0;margin-top:5px;clear:both;margin-bottom:0;padding:0;border:0;height:auto;width:auto}#onetrust-banner-sdk.ot-iab-2 #onetrust-button-group button{display:block}#onetrust-banner-sdk.ot-close-btn-link{padding-top:25px}#onetrust-banner-sdk.ot-close-btn-link #onetrust-close-btn-container{top:15px;transform:none;right:15px}#onetrust-banner-sdk.ot-close-btn-link #onetrust-close-btn-container button{padding:0;white-space:pre-wrap;border:none;height:auto;line-height:1.5;text-decoration:underline;font-size:.69em}#onetrust-banner-sdk #onetrust-policy-text,#onetrust-banner-sdk .ot-dpd-desc,#onetrust-banner-sdk .ot-b-addl-desc{font-size:.813em;line-height:1.5}#onetrust-banner-sdk .ot-dpd-desc{margin-bottom:10px}#onetrust-banner-sdk .ot-dpd-desc>.ot-b-addl-desc{margin-top:10px;margin-bottom:10px;font-size:1em}@media only screen and (max-width: 425px){#onetrust-banner-sdk #onetrust-close-btn-container{position:absolute;top:6px;right:2px}#onetrust-banner-sdk #onetrust-policy{margin-left:0;margin-top:3em}#onetrust-banner-sdk #onetrust-button-group{display:block}#onetrust-banner-sdk #onetrust-accept-btn-handler,#onetrust-banner-sdk #onetrust-reject-all-handler,#onetrust-banner-sdk #onetrust-pc-btn-handler{width:100%}#onetrust-banner-sdk .onetrust-close-btn-ui{top:auto;transform:none}#onetrust-banner-sdk #onetrust-policy-title{display:inline;float:none}#onetrust-banner-sdk #banner-options{margin:0;padding:0;width:100%}}@media only screen and (max-width: 550px){#onetrust-button-group.ot-button-order-container #onetrust-accept-btn-handler,#onetrust-button-group.ot-button-order-container #onetrust-reject-all-handler,#onetrust-button-group.ot-button-order-container #onetrust-pc-btn-handler{margin-right:0}#onetrust-banner-sdk .has-reject-all-button div#onetrust-button-group.ot-button-order-container #onetrust-accept-btn-handler,#onetrust-banner-sdk .has-reject-all-button div#onetrust-button-group.ot-button-order-container #onetrust-reject-all-handler,#onetrust-banner-sdk .has-reject-all-button div#onetrust-button-group.ot-button-order-container #onetrust-pc-btn-handler{margin-right:0}}@media only screen and (min-width: 426px)and (max-width: 896px){#onetrust-banner-sdk #onetrust-close-btn-container{position:absolute;top:0;right:0}#onetrust-banner-sdk #onetrust-policy{margin-left:1em;margin-right:1em}#onetrust-banner-sdk .onetrust-close-btn-ui{top:10px;right:10px}#onetrust-banner-sdk:not(.ot-iab-2) #onetrust-group-container{width:95%}#onetrust-banner-sdk.ot-iab-2 #onetrust-group-container{width:100%}#onetrust-banner-sdk.ot-bnr-w-logo #onetrust-button-group-parent{padding-left:50px}#onetrust-banner-sdk #onetrust-button-group-parent{width:100%;position:relative;margin-left:0}#onetrust-banner-sdk #onetrust-button-group button{display:inline-block}#onetrust-banner-sdk #onetrust-button-group{margin-right:0;text-align:center}#onetrust-banner-sdk #onetrust-button-group.ot-button-order-container #onetrust-accept-btn-handler,#onetrust-banner-sdk #onetrust-button-group.ot-button-order-container #onetrust-reject-all-handler,#onetrust-banner-sdk #onetrust-button-group.ot-button-order-container #onetrust-pc-btn-handler{width:auto}#onetrust-banner-sdk .has-reject-all-button #onetrust-button-group.ot-button-order-container{display:inline-flex;flex-wrap:wrap}#onetrust-banner-sdk .has-reject-all-button #onetrust-button-group.ot-button-order-container #onetrust-pc-btn-handler,#onetrust-banner-sdk .has-reject-all-button #onetrust-button-group.ot-button-order-container #onetrust-reject-all-handler,#onetrust-banner-sdk .has-reject-all-button #onetrust-button-group.ot-button-order-container #onetrust-accept-btn-handler{float:none}#onetrust-banner-sdk .has-reject-all-button #onetrust-button-group.ot-button-order-container *[class*=ot-button-order-]:nth-of-type(1){margin-right:auto !important}#onetrust-banner-sdk .has-reject-all-button #onetrust-pc-btn-handler{float:left}#onetrust-banner-sdk .has-reject-all-button #onetrust-reject-all-handler,#onetrust-banner-sdk .has-reject-all-button #onetrust-accept-btn-handler{float:right}#onetrust-banner-sdk .has-reject-all-button #onetrust-button-group{width:calc(100% - 2em);margin-right:0}#onetrust-banner-sdk .has-reject-all-button #onetrust-pc-btn-handler.cookie-setting-link{padding-left:0px;text-align:left}#onetrust-banner-sdk.ot-buttons-fw .ot-sdk-three button{width:100%;text-align:center}#onetrust-banner-sdk.ot-buttons-fw #onetrust-button-group-parent button{float:none}#onetrust-banner-sdk.ot-buttons-fw #onetrust-pc-btn-handler.cookie-setting-link{text-align:center}}@media only screen and (min-width: 550px){#onetrust-banner-sdk .banner-option:not(:first-child){border-left:1px solid #d8d8d8;padding-left:25px}}@media only screen and (min-width: 425px)and (max-width: 550px){#onetrust-banner-sdk.ot-iab-2 #onetrust-button-group,#onetrust-banner-sdk.ot-iab-2 .banner-option{width:100%}#onetrust-banner-sdk.ot-iab-2 #onetrust-button-group #onetrust-accept-btn-handler,#onetrust-banner-sdk.ot-iab-2 #onetrust-button-group #onetrust-reject-all-handler,#onetrust-banner-sdk.ot-iab-2 #onetrust-button-group #onetrust-pc-btn-handler{width:100%}#onetrust-banner-sdk.ot-iab-2 #onetrust-button-group #onetrust-accept-btn-handler,#onetrust-banner-sdk.ot-iab-2 #onetrust-button-group #onetrust-reject-all-handler{float:left}#onetrust-banner-sdk.ot-iab-2 #onetrust-button-group.ot-button-order-container{width:auto}}@media only screen and (min-width: 769px){#onetrust-banner-sdk #onetrust-button-group{margin-right:30%}#onetrust-banner-sdk #banner-options{margin-left:2em;margin-right:5em;margin-bottom:1.25em;width:calc(100% - 7em)}}@media only screen and (min-width: 897px)and (max-width: 1023px){#onetrust-banner-sdk.vertical-align-content #onetrust-button-group-parent{position:absolute;top:50%;left:80%;transform:translateY(-50%)}#onetrust-banner-sdk #onetrust-close-btn-container{top:50%;margin:auto;transform:translate(-50%, -50%);position:absolute;padding:0;right:0}#onetrust-banner-sdk #onetrust-close-btn-container button{position:relative;margin:0;right:-22px;top:2px}}@media only screen and (min-width: 1024px){#onetrust-banner-sdk #onetrust-close-btn-container{top:50%;margin:auto;transform:translate(-50%, -50%);position:absolute;right:0}#onetrust-banner-sdk #onetrust-close-btn-container button{right:-12px}#onetrust-banner-sdk #onetrust-policy{margin-left:2em}#onetrust-banner-sdk.vertical-align-content #onetrust-button-group-parent{position:absolute;top:50%;left:60%;transform:translateY(-50%)}#onetrust-banner-sdk .ot-optout-signal{width:50%}#onetrust-banner-sdk.ot-iab-2 #onetrust-policy-title{width:50%}#onetrust-banner-sdk.ot-iab-2 #onetrust-policy-text,#onetrust-banner-sdk.ot-iab-2 :not(.ot-dpd-desc)>.ot-b-addl-desc{margin-bottom:1em;width:50%;border-right:1px solid #d8d8d8;padding-right:1rem}#onetrust-banner-sdk.ot-iab-2 #onetrust-policy-text{margin-bottom:0;padding-bottom:1em}#onetrust-banner-sdk.ot-iab-2 :not(.ot-dpd-desc)>.ot-b-addl-desc{margin-bottom:0;padding-bottom:1em}#onetrust-banner-sdk.ot-iab-2 .ot-dpd-container{width:45%;padding-left:1rem;display:inline-block;float:none}#onetrust-banner-sdk.ot-iab-2 .ot-dpd-title{line-height:1.7}#onetrust-banner-sdk.ot-iab-2 #onetrust-button-group-parent{left:auto;right:4%;margin-left:0}#onetrust-banner-sdk.ot-iab-2 #onetrust-button-group button{display:block}#onetrust-banner-sdk:not(.ot-iab-2) #onetrust-button-group-parent{margin:auto;width:30%}#onetrust-banner-sdk:not(.ot-iab-2) #onetrust-group-container{width:60%}#onetrust-banner-sdk #onetrust-button-group{margin-right:auto}#onetrust-banner-sdk #onetrust-accept-btn-handler,#onetrust-banner-sdk #onetrust-reject-all-handler,#onetrust-banner-sdk #onetrust-pc-btn-handler{margin-top:1em}}@media only screen and (min-width: 890px){#onetrust-banner-sdk.ot-buttons-fw:not(.ot-iab-2) #onetrust-button-group-parent{padding-left:3%;padding-right:4%;margin-left:0}#onetrust-banner-sdk.ot-buttons-fw:not(.ot-iab-2) #onetrust-button-group{margin-right:0;margin-top:1.25em;width:100%}#onetrust-banner-sdk.ot-buttons-fw:not(.ot-iab-2) #onetrust-button-group button{width:100%;margin-bottom:5px;margin-top:5px}#onetrust-banner-sdk.ot-buttons-fw:not(.ot-iab-2) #onetrust-button-group button:last-of-type{margin-bottom:20px}}@media only screen and (min-width: 1280px){#onetrust-banner-sdk:not(.ot-iab-2) #onetrust-group-container{width:55%}#onetrust-banner-sdk:not(.ot-iab-2) #onetrust-button-group-parent{width:44%;padding-left:2%;padding-right:2%}#onetrust-banner-sdk:not(.ot-iab-2).vertical-align-content #onetrust-button-group-parent{position:absolute;left:55%}}"
}

In [45]:
import base64

def b64decode(data):
    return base64.b64decode(data).decode('utf-8')

html = b64decode(x['html'])
html

'<div id="onetrust-banner-sdk" class="otFlat"><div role="dialog"><div class="ot-sdk-container"><div class="ot-sdk-row"><div id="onetrust-group-container" class="ot-sdk-eight ot-sdk-columns"><div class="banner_logo"></div><div id="onetrust-policy"><h2 id="onetrust-policy-title">Title</h2><div id="onetrust-policy-text">title<a href="#">policy</a></div><div class="ot-dpd-container"><h3 class="ot-dpd-title">We collect data in order to provide:</h3><div class="ot-dpd-content"><p class="ot-dpd-desc">description</p></div></div></div></div><div id="onetrust-button-group-parent" class="ot-sdk-three ot-sdk-columns"><div id="onetrust-button-group"><button id="onetrust-pc-btn-handler">choice</button> <button id="onetrust-reject-all-handler">Reject</button> <button id="onetrust-accept-btn-handler">Accept</button></div></div></div></div><div class="ot-sdk-container" id="banner-options"><div class="ot-sdk-four ot-sdk-columns banner-option"><button aria-expanded="false" class="banner-option-input"><sp

In [54]:
j = rsp.json()

In [55]:
disk.write_json('data/_v2/leaguegamelog_24_25.json', j)

In [56]:
list(j)

['resource', 'parameters', 'resultSets']

In [57]:
a = j['resultSets'][-1]
a = pd.DataFrame(a['rowSet'], columns=a['headers'])
a

,SEASON_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,...,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,FANTASY_PTS,VIDEO_AVAILABLE
0,22024,1628436,Luke Kornet,1610612738,BOS,Boston Celtics,0022401187,2025-04-13,BOS vs. CHA,W,...,8,2,1,4,0,2,11,1,38.6,1
1,22024,1630202,Payton Pritchard,1610612738,BOS,Boston Celtics,0022401187,2025-04-13,BOS vs. CHA,W,...,7,7,1,0,4,0,34,8,51.9,1
2,22024,202684,Tristan Thompson,1610612739,CLE,Cleveland Cavaliers,0022401189,2025-04-13,CLE vs. IND,L,...,20,1,1,2,3,3,3,-25,34.5,1
3,22024,1631288,Jamal Cain,1610612740,NOP,New Orleans Pelicans,0022401196,2025-04-13,NOP vs. OKC,L,...,4,1,1,0,1,4,18,-4,26.3,1
4,22024,1628989,Kevin Huerter,1610612741,CHI,Chicago Bulls,0022401191,2025-04-13,CHI @ PHI,W,...,2,2,0,0,0,2,18,4,23.4,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26301,22024,1630574,Ariel Hukporti,1610612752,NYK,New York Knicks,0022400061,2024-10-22,NYK @ BOS,L,...,4,0,0,1,0,0,0,9,7.8,1
26302,22024,1631210,Jacob Toppin,1610612752,NYK,New York Knicks,0022400061,2024-10-22,NYK @ BOS,L,...,0,0,0,0,0,0,0,0,0.0,1
26303,22024,1642359,Pacôme Dadiet,1610612752,NYK,New York Knicks,0022400061,2024-10-22,NYK @ BOS,L,...,1,0,0,0,0,1,3,10,4.2,1
26304,22024,1629637,Jaxson Hayes,1610612747,LAL,Los Angeles Lakers,0022400062,2024-10-22,LAL vs. MIN,W,...,4,0,0,1,0,3,10,-1,17.8,1


In [31]:
a.PERSON_ID.value_counts()

PERSON_ID
1628427    1
76001      1
76002      1
76003      1
51         1
          ..
1630173    1
203518     1
76007      1
76006      1
76005      1
Name: count, Length: 5126, dtype: int64

In [ ]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(rsp.text, 'html.parser')

In [ ]:
from IPython.display import display, HTML


In [ ]:
contents = soup.find_all(attrs={'type': 'application/json'})

In [1]:
import json

json.loads(contents[0].text)

NameError: name 'contents' is not defined

In [5]:
import requests
from bs4 import BeautifulSoup

In [6]:
header_contents = """
accept
*/*
accept-encoding
gzip, deflate, br, zstd
accept-language
nb-NO,nb;q=0.9,no;q=0.8,nn;q=0.7,en-US;q=0.6,en;q=0.5
cache-control
no-cache
cookie
SWID=C14BDF1F-F1BF-4D0E-CDA7-AB4F6B41A5B9; OptanonAlertBoxClosed=2026-04-10T00:46:09.298Z; eupubconsent-v2=CQicU3AQicU3AAcABBENCZFgAAAAAEPgAChQAAAYiABMNDogjLIgUCBQEIIEACgrCACgQBAAAkDRAQAmDAhyBgAusJkAIAUAAwQAgABBgACAAASABCIAIACAQAgQCBQABgAQBAQAMDAAGACxEAgABAdAxTAggECwASIyqDTAlAASCAlsqEEoGBBXCFIscAggREwUAAAIABQAAID4WAhJKCViQQBcQXQAAEAAAUQIkCKQswBBUGaLQVgScBkaYBk-YJklOgyAJghIyDIhNUEg8UxRCghyA2KWYA6eIKAGXayQh_qAAAAA.YAAACHwAAAAA.IMStR_G__bXlv-bb36btkeYxf9_hr7sQxBgbJs24FzLvW7JwH32E7NEzatqYKmRIAu3TBIQNtHJjURUChKIgVrzDsaE2U4TtKJ-BkiHMZY2tYCFxvm4tjWQCZ4ur_51d9mT-t7dr-2dzy27hnv3a9fuS1UJidKYetHfv8ZBOT-_IU9_x-_4v4_MbpEm-eS1v_tWtt43d64vP_dpuxt-Tyff7____73_e7X__e__33_-qXX_77____________f__________9oAA; s_ecid=MCMID%7C86802923025445614922182749332434390880; userZip=1534; tveMVPDAuth=; _gcl_au=1.1.301185680.1775792131; tveAuth=; espn_aid=e6bc25f6-d240-4ce9-bd8d-d6debaf7961f; country=no; hashedIp=f59d3283d257c8367fe84395284b8f0cfa5626cd5840aff70f1329a2efe7870c; block.check=false%7Cfalse; espn-prev-page=espn%3Anba%3Agame%3Aboxscore; AMCVS_EE0201AC512D2BE80A490D4C%40AdobeOrg=1; AMCV_EE0201AC512D2BE80A490D4C%40AdobeOrg=-50417514%7CMCMID%7C86802923025445614922182749332434390880%7CMCAAMLH-1776401056%7C6%7CMCAAMB-1776401056%7C6G1ynYcLPuiQxYZrsz_pkqfLG9yMXBpb2zX5dvJdYQJzPXImdj0y%7CMCOPTOUT-1775803456s%7CNONE%7CMCAID%7CNONE%7CvVersion%7C5.5.0; s_cc=true; __eoi=ID=47428297a185a554:T=1775792131:RT=1775796257:S=AA-AfjbmjXeqfHnLkshSjZoku7Bq; s_ensNR=1775796614169-Repeat; OptanonConsent=isGpcEnabled=0&datestamp=Fri+Apr+10+2026+06%3A50%3A14+GMT%2B0200+(Central+European+Summer+Time)&version=202601.2.0&browserGpcFlag=0&isIABGlobal=false&identifierType=DeviceId&hosts=&consentId=71014b4a-9693-4aff-8378-db4fa126c9a0&interactionCount=1&isAnonUser=1&prevHadToken=0&landingPath=NotLandingPage&groups=C0001%3A1%2CC0002%3A0%2CC0003%3A0%2CC0004%3A0%2CV2STACK1%3A0%2CV2STACK42%3A0%2CBG397%3A1&iType=2&intType=2&crTime=1775781969818&geolocation=GB%3B&AwaitingReconsent=false; s_sq=%5B%5BB%5D%5D
pragma
no-cache
referer
https://www.espn.com/
sec-ch-ua
"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"
sec-ch-ua-mobile
?0
sec-ch-ua-platform
"Windows"
sec-fetch-dest
script
sec-fetch-mode
no-cors
sec-fetch-site
same-site
user-agent
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36
""".strip()

headers = {}
for idx, line in enumerate(header_contents.splitlines()):
    if idx % 2 == 0:
        key = line.strip()
    else:
        value = line.strip()
        headers[key] = value

In [7]:
url = 'https://www.espn.com/nba/boxscore/_/gameId/401811025'
rsp = requests.get(url, headers=headers)
rsp.status_code

200

In [8]:
soup = BeautifulSoup(rsp.text, 'html.parser')

In [16]:
s = soup.find('script', string=lambda s: s and '__CONFIG__' in s)

In [18]:
import re
import json

text = s.string
config_match = re.search(r"window\['__CONFIG__'\]\s*=\s*(\{.*?\});", text)
espn_match = re.search(r"window\['__espnfitt__'\]\s*=\s*(\{.*?\});", text)

config = json.loads(config_match.group(1))
espn = json.loads(espn_match.group(1))

In [ ]:
# with open(f'/home/abraham/uni/ikt453/project/v1/data/espn___CONFIG__.json', 'w') as f:
#     json.dump(config, f, indent=4)

In [ ]:
from IPython.display import display, HTML

# display(HTML(rsp.text))